# 🌌 Space Odyssey — GRPO Training (Kaggle Edition)

**Before running:**
1. Enable GPU: Settings (⚙️ right panel) → Accelerator → **GPU T4 x2**
2. Enable Internet: Settings → Internet → **ON**
3. Run all cells in order — no restarts needed on Kaggle!

**Expected time:** ~2-3 hours on T4 x2

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL
# On Kaggle, no restart is needed after install!
# ══════════════════════════════════════════════════════════════════════
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

pip('unsloth[kaggle-new]', '--no-deps')
pip('xformers', 'trl', 'peft', 'accelerate', 'bitsandbytes')
pip('transformers', 'datasets', 'tokenizers')
pip('gymnasium', 'matplotlib', 'plotly')

print('✅ Installation complete!')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — CLONE REPO
# ══════════════════════════════════════════════════════════════════════
import os, sys

REPO_DIR = '/kaggle/working/Space-Odyssey'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/lokendra005/Space-Odyssey.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main
    print('Already cloned, pulled latest.')

# Add repo to Python path
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print('\n✅ Working directory:', os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3 — DIAGNOSTIC CHECK
# ══════════════════════════════════════════════════════════════════════
import importlib

REQUIRED = ['torch', 'unsloth', 'trl', 'peft', 'transformers', 'datasets', 'bitsandbytes']
all_ok = True
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        print(f'  ✅ {pkg:<18} {getattr(mod, "__version__", "ok")}')
    except ImportError as e:
        print(f'  ❌ {pkg:<18} MISSING — {e}')
        all_ok = False

import torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    vram = round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {vram} GB')

if not all_ok:
    print('\n⚠️  Some packages missing — re-run Cell 1')
elif n_gpu == 0:
    print('\n⚠️  No GPU! Enable it in Settings → Accelerator')
else:
    print(f'\n✅ All good! {n_gpu} GPU(s) ready. Proceed to Cell 4.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — ENVIRONMENT SANITY CHECK
# ══════════════════════════════════════════════════════════════════════
from environment.station_env import ProcurementDriftEnv
from training.reward import is_proposal_dangerous, compute_violation_severity

env = ProcurementDriftEnv()
obs, info = env.reset(seed=42)
state = env._flat_state()
proposal = env.current_proposal

print('--- Station Telemetry (seed=42) ---')
print(f'  O2   = {state["oxygen"]:.1f}%')
print(f'  Power= {state["power"]:.1f}%')
print(f'  Hull = {state["hull_integrity"]:.1f}%')
print(f'  Proposal: {proposal["type"]} | label={proposal["risk_level"]}')
print(f'  Dangerous? {is_proposal_dangerous(state, proposal)}')
env.close()
print('\n✅ Environment healthy!')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — PHASE 1: SFT REASONING WARMUP (~8 min)
# ══════════════════════════════════════════════════════════════════════

# Unsloth MUST be imported first
import unsloth
from training.sft_warmup import _HAS_TRAIN_DEPS, _TRAIN_DEPS_ERROR, run_sft

if not _HAS_TRAIN_DEPS:
    raise RuntimeError(f'Missing dep: {_TRAIN_DEPS_ERROR}\nRe-run Cell 1.')

print('🧠 Phase 1: Reasoning Warmup starting...')
model, tokenizer = run_sft()
print('\n✅ Phase 1 done! → overseer_lora_warmup/')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — PHASES 2-4: GRPO CURRICULUM RL (~2-3 hrs on T4 x2)
# ══════════════════════════════════════════════════════════════════════
import gc, torch, importlib

# Free SFT model from VRAM before loading GRPO
try: del model, tokenizer
except: pass
gc.collect(); torch.cuda.empty_cache()
print(f'✅ GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Fresh import (avoids stale module state)
import training.grpo_train as _grpo
importlib.reload(_grpo)
from training.grpo_train import run_grpo_training

print('\n🛡️ GRPO Curriculum RL starting...')
print('   Watch reward rise toward positive values = model is learning!')
print('-' * 50)

run_grpo_training(
    sft_adapter_path='overseer_lora_warmup',
    output_dir='overseer_grpo_final',
    easy_episodes=20,
    hard_episodes=50,
    precision_episodes=30,
)

print('\n🏆 TRAINING COMPLETE! → overseer_grpo_final/')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 — EVALUATION & PLOTS
# ══════════════════════════════════════════════════════════════════════
import os
os.makedirs('assets', exist_ok=True)

!PYTHONPATH=/kaggle/working/Space-Odyssey python eval/evaluate.py
!PYTHONPATH=/kaggle/working/Space-Odyssey python eval/plot_training_curves.py

from IPython.display import Image, display
plots = [
    ('assets/training_curve.png',       'Training Reward Curve'),
    ('assets/violation_prevention.png', 'Violation Prevention Rate'),
    ('assets/eval_results.png',         'Trained vs Baseline'),
    ('assets/reward_matrix.png',        'Reward Matrix'),
]
for path, title in plots:
    if os.path.exists(path):
        print(f'\n── {title} ──')
        display(Image(path))
    else:
        print(f'[skip] {path} not found')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8 — PACKAGE & SAVE OUTPUT
# On Kaggle, files in /kaggle/working/ are auto-saved as output.
# You can also download directly from the Output tab.
# ══════════════════════════════════════════════════════════════════════
import shutil, os

if os.path.isdir('overseer_grpo_final'):
    shutil.make_archive('/kaggle/working/trained_brain', 'zip', '.', 'overseer_grpo_final')
    size_mb = round(os.path.getsize('/kaggle/working/trained_brain.zip') / 1e6, 1)
    print(f'✅ Saved → /kaggle/working/trained_brain.zip ({size_mb} MB)')
    print('   Download from the Output tab (▶ right panel) when done.')
else:
    print('⚠️ overseer_grpo_final/ not found — did Cell 6 complete?')

# Also copy plots as Kaggle output
for f in ['assets/training_curve.png', 'assets/eval_results.png', 'assets/violation_prevention.png']:
    if os.path.exists(f):
        shutil.copy(f, f'/kaggle/working/{os.path.basename(f)}')
        print(f'  📊 {f} saved to output')